In [18]:
from utils import build_index
from dotenv import load_dotenv
load_dotenv()
import json

In [2]:
file = "youtube_transcripts/dataset.json"

In [3]:
with open(file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data)

[{'question': 'What is emotional intelligence?', 'answer': 'Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.', 'video_id': 'BqF50IuR3_c'}, {'question': 'What are the four main domains of emotional intelligence?', 'answer': 'The four domains are self-awareness, self-management, empathy, and relationship management.', 'video_id': 'BqF50IuR3_c'}, {'question': 'What does it mean to be self-aware?', 'answer': 'Being self-aware means recognizing what you are feeling and understanding why you are feeling it.', 'video_id': 'BqF50IuR3_c'}, {'question': 'How can self-awareness improve your intuition?', 'answer': 'Understanding your emotions and their causes gives you useful internal information that can support your intuition.', 'video_id': 'BqF50IuR3_c'}, {'question': 'How can self-awareness help you make better decisions?', 'answer': 'Recognizing your feelings helps you understand how they may be influencing your judgment a

In [4]:

index = build_index(data)

In [5]:
question = "How can I improve my self-awareness and emotional intelligence?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'question': 'What is emotional intelligence?',
  'answer': 'Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.',
  'video_id': 'BqF50IuR3_c'},
 {'question': 'How can self-awareness improve your intuition?',
  'answer': 'Understanding your emotions and their causes gives you useful internal information that can support your intuition.',
  'video_id': 'BqF50IuR3_c'},
 {'question': 'How can recording yourself improve self-awareness?',
  'answer': 'Watching a recording can reveal gestures, posture, tone, and speaking habits that you did not realize you had.',
  'video_id': 'ln8rIBZbWAE'},
 {'question': 'How is emotional intelligence connected to everyday life?',
  'answer': 'Emotional intelligence affects how effectively you manage yourself and interact with other people.',
  'video_id': 'Y7m9eNoB3NU'},
 {'question': 'What is the difference between private and public self-awareness?',
  'answer': 'Private self-awareness

In [10]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("video_URL: https://www.youtube.com/watch?v=" + doc["video_id"]+ " ")

    return "\n".join(lines).strip()

In [11]:
build_context(search_results)

'Q: What is emotional intelligence?\nA: Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.\nvideo_URL: https://www.youtube.com/watch?v=BqF50IuR3_c \nQ: How can self-awareness improve your intuition?\nA: Understanding your emotions and their causes gives you useful internal information that can support your intuition.\nvideo_URL: https://www.youtube.com/watch?v=BqF50IuR3_c \nQ: How can recording yourself improve self-awareness?\nA: Watching a recording can reveal gestures, posture, tone, and speaking habits that you did not realize you had.\nvideo_URL: https://www.youtube.com/watch?v=ln8rIBZbWAE \nQ: How is emotional intelligence connected to everyday life?\nA: Emotional intelligence affects how effectively you manage yourself and interact with other people.\nvideo_URL: https://www.youtube.com/watch?v=Y7m9eNoB3NU \nQ: What is the difference between private and public self-awareness?\nA: Private self-awareness is unders

In [12]:
INSTRUCTIONS = """
Your task is to be a self-awareness and emotional intelligence friendly assistant. You will be provided with a question based on the provided context.

Use the context to find relevant information and provide accurate
answers. 
Also use the links on video_URL to provide additional information and resources to the user.

If the answer is not found in the context,
respond with "I don't know."
"""

In [21]:
PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context} 
""".strip()

In [22]:
def build_prompt( query, search_results):
    """Render the prompt template using a query and search context.

    Args:
        query (str): The user's question.
        search_results (list): Results returned by `search()`.

    Returns:
        str: The rendered prompt ready to send to the LLM.
    """
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(
        question=query, context=context
    )

In [23]:
prompt = build_prompt(question, search_results)
prompt

'QUESTION: How can I improve my self-awareness and emotional intelligence?\n\nCONTEXT:\nQ: What is emotional intelligence?\nA: Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.\nvideo_URL: https://www.youtube.com/watch?v=BqF50IuR3_c \nQ: How can self-awareness improve your intuition?\nA: Understanding your emotions and their causes gives you useful internal information that can support your intuition.\nvideo_URL: https://www.youtube.com/watch?v=BqF50IuR3_c \nQ: How can recording yourself improve self-awareness?\nA: Watching a recording can reveal gestures, posture, tone, and speaking habits that you did not realize you had.\nvideo_URL: https://www.youtube.com/watch?v=ln8rIBZbWAE \nQ: How is emotional intelligence connected to everyday life?\nA: Emotional intelligence affects how effectively you manage yourself and interact with other people.\nvideo_URL: https://www.youtube.com/watch?v=Y7m9eNoB3NU \nQ: What is the dif

In [24]:
from openai import OpenAI

llm_client = OpenAI()
model="gpt-5.4-mini"

def llm(prompt):
    """Call the LLM client with developer instructions and a user prompt.

    Args:
        prompt (str): The prompt text to send as the user message.

    Returns:
        str: The model's textual response.
    """
    input_messages = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]

    response = llm_client.responses.create(
        model=model,
        input=input_messages
    )

    return response.output_text

In [25]:
answer = llm(prompt)
answer

'You can improve self-awareness by understanding your internal experience—especially your emotions and what causes them. That gives you useful information about yourself, and it can also support your intuition.\n\nYou can also improve self-awareness by recording yourself and reviewing it. Watching a recording may reveal gestures, posture, tone, and speaking habits you didn’t notice before.\n\nFor emotional intelligence, focus on how effectively you handle yourself, your emotions, and your relationships with other people. It affects how well you manage yourself and interact with others in everyday life.\n\nA helpful way to think about self-awareness is:\n- **Private self-awareness:** understanding your internal experience\n- **Public self-awareness:** understanding how other people perceive you\n\nRelated videos:\n- https://www.youtube.com/watch?v=BqF50IuR3_c\n- https://www.youtube.com/watch?v=ln8rIBZbWAE\n- https://www.youtube.com/watch?v=Y7m9eNoB3NU'